# BANKING77: TF-IDF and LAC routing

BANKING77 contains customer-support requests labelled with 77 banking intents,
such as locating a card or asking about a cash withdrawal. This small demo turns
a request into a shortlist of possible intents, then automates only when one
intent remains. We use a lightweight text classifier and the project's existing
LAC functions—no language model is needed.

Run from this notebook's directory with the repository installed:
`pip install -e ".[example,notebook]"` from the repository root provides the
packages needed here. The dataset should be in `data/raw/banking77`; see the
[data instructions](../../data/README.md) if it is not available.

In [1]:
from pathlib import Path

import numpy as np

from llm_scorekit.calibration import conformal_quantile
from llm_scorekit.data import load_banking77
from llm_scorekit.metrics import (
    automated_error_rate,
    automation_rate,
    average_set_size,
    empirical_coverage,
)
from llm_scorekit.models.baseline import fit_tfidf_classifier
from llm_scorekit.prediction_sets import lac_prediction_sets
from llm_scorekit.scores import lac_scores
from llm_scorekit.selection import singleton_mask

## 1. Keep training, calibration and test separate

The loader splits the official training data into 75% training and 25% calibration, keeping the intent proportions similar. The official test data stays separate: we will use it only to inspect the finished pipeline.

In [2]:
data = load_banking77(Path("../../data/raw/banking77"), random_seed=42)
class_names = data.class_names
alpha = 0.10

print("Training requests:", len(data.train.texts))
print("Calibration requests:", len(data.calibration.texts))
print("Official test requests:", len(data.test.texts))
print("Possible intents:", len(class_names))

Training requests: 7502
Calibration requests: 2501
Official test requests: 3080
Possible intents: 77


## 2. Fit a simple text classifier

**TF-IDF** represents a request using words and two-word phrases. Terms that are frequent in a request but less common across training requests receive more weight. Logistic regression converts those features into probabilities over the 77 intents.

Both the vocabulary and classifier are learned only from the training partition. These model probabilities are inputs to LAC, not a guarantee that an individual prediction is correct.

In [3]:
# Learn text features and the classifier only from training requests.
model = fit_tfidf_classifier(data.train)

# Align probability columns with the label IDs used by the data loader.
label_to_column = {label: column for column, label in enumerate(model.classes_)}
column_order = [label_to_column[label] for label in range(len(class_names))]
print("Classifier fitted on the training partition.")

Classifier fitted on the training partition.


## 3. Calibrate LAC once

LAC gives each calibration request the score `1 - probability of its true intent`. A higher score means the model gave less probability to the correct answer.

The finite-sample conformal quantile gives one threshold for our fixed `alpha = 0.10`. A new intent enters the set when `1 - its probability <= threshold`. Calibration changes the threshold, not the fitted classifier.

In [4]:
calibration_probabilities = model.predict_proba(data.calibration.texts)
calibration_probabilities = calibration_probabilities[:, column_order]
calibration_scores = lac_scores(calibration_probabilities, data.calibration.labels)
threshold = conformal_quantile(calibration_scores, alpha)

print(f"Target marginal coverage: {1 - alpha:.0%}")
print(f"LAC score threshold: {threshold:.4f}")
print(f"Include intents with probability at least: {1 - threshold:.4f}")

Target marginal coverage: 90%
LAC score threshold: 0.9405
Include intents with probability at least: 0.0595


## 4. Look at one request and its routing decision

We now apply the fixed model and threshold to the official test requests. A **singleton** set contains exactly one intent: our policy automates that intent. An empty set or a set with several intents is deferred for review. This is a demonstration of the decision rule; it does not send requests anywhere.

In [5]:
test_probabilities = model.predict_proba(data.test.texts)
test_probabilities = test_probabilities[:, column_order]
prediction_sets = lac_prediction_sets(test_probabilities, threshold)
automated = singleton_mask(prediction_sets)
predictions = np.argmax(test_probabilities, axis=1)

# Change this index to inspect another official test request.
request_index = 0
probabilities = test_probabilities[request_index]
top_indices = np.argsort(-probabilities)[:3]
set_indices = np.flatnonzero(prediction_sets[request_index])
set_intents = [class_names[index] for index in set_indices]

print("Request:", data.test.texts[request_index])
print("True intent:", class_names[data.test.labels[request_index]])
print("Top three model probabilities:")
for index in top_indices:
    print(f"  {class_names[index]}: {probabilities[index]:.3f}")
print("LAC set:", set_intents)

if automated[request_index]:
    print("Decision: automate ->", set_intents[0])
else:
    print("Decision: defer for review")

Request: How do I locate my card?
True intent: card_arrival
Top three model probabilities:
  card_linking: 0.051
  get_physical_card: 0.047
  activate_my_card: 0.044
LAC set: []
Decision: defer for review


## 5. Summarize the held-out requests

Coverage is the fraction whose set contains the true intent; average size describes how many intents remain. Automation is the fraction with singleton sets, and automated error counts mistakes **only among those singletons**. This summary is a descriptive result for this one fixed pipeline, not a comparison of methods.

In [6]:
coverage = empirical_coverage(prediction_sets, data.test.labels)
mean_size = average_set_size(prediction_sets)
automation = automation_rate(automated)
automated_error = automated_error_rate(predictions, data.test.labels, automated)

# Error has no denominator when there are no automated requests.
if automated.any():
    error_text = f"{automated_error:.1%}"
else:
    error_text = "N/A (no automated requests)"

print(f"Official test requests: {len(data.test.texts)}")
print(f"Prediction-set coverage: {coverage:.1%}")
print(f"Average set size: {mean_size:.2f}")
print(f"Automation rate: {automation:.1%} ({automated.sum()} requests)")
print(f"Error among automated requests: {error_text}")

Official test requests: 3080
Prediction-set coverage: 89.5%
Average set size: 1.52
Automation rate: 54.7% (1686 requests)
Error among automated requests: 5.4%


## Takeaway

The classifier supplies probabilities, separate calibration examples set the LAC
threshold, and the singleton policy decides whether to automate or defer.
The test results show why **coverage and routing accuracy are different**:
keeping the correct answer in the shortlist is not the same as choosing it
automatically.

The 90% coverage target is not a promise for every run or every intent, nor a
90% accuracy guarantee for automatic decisions. Its statistical guarantee
assumes calibration and future requests follow the same unchanged distribution;
an official train/test split does not establish that assumption by itself.

This notebook illustrates one pipeline. The
[full BANKING77 study](../../docs/banking77/baseline/tfidf_lac.md) examines several
settings and training splits, alongside a confidence-threshold baseline.